# Configure workspace

**Author:** Juliana Thomasia C. Marcos\
**Input:** Databricks workspace configuration parameters and Roboflow API key\
**Output:** Initialized images and annotations tables, secret scope, and access permissions

## Overview

This notebook initializes the Databricks workspace configuration required to use the enablement package and its associated MLOps workflows. It sets up the relevant catalog and schema information and creates the images and annotations tables using predefined JSON schemas, providing a consistent structure for storing and managing imagery and annotation data.

The notebook also configures a Databricks secret scope for securely storing the Roboflow API key used to interact with Roboflow services. It checks for an existing scope and secret before attempting creation and grants the specified user read access to the scope's secrets. This configuration prepares the workspace for subsequent data processing, model development, and inference workflows.

# Packages import and installation

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import AclPermission
from pyspark.sql import SparkSession

from ag_vision.data_io.schemas import create_table_from_schema

In [0]:
spark = SparkSession.builder.getOrCreate()

# Workspace variables set up

In [0]:
catalog_name = "use1_prod_artemis_catalog_3718194974443840"
schema = "production_test"
volume = "data_test"

In [0]:
workspace_schema_name = ""

if catalog_name:
    workspace_schema_name = f"{catalog_name}.{schema}"
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {workspace_schema_name}")
else:
    print(f"The catalog name is missing.")

In [0]:
if workspace_schema_name:
    workspace_volume_name = f"{workspace_schema_name}.{volume}"
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {workspace_volume_name}")
else:
    print(f"There is no workspace schema name.")

# Images table creation

In [0]:
if catalog_name:
    images_table_name = "images_draft"
    images_table = f"{catalog_name}.{schema}.{images_table_name}"
    create_table_from_schema(spark, images_table, "images_table.json")

# Annotations table creation

In [0]:
if catalog_name:
    annotations_table_name = "annotations_draft"
    annotations_table = f"{catalog_name}.{schema}.{annotations_table_name}"
    create_table_from_schema(spark, annotations_table, "annotations_table.json")

# Enablement package scope and secrets

In [0]:
list_scope_names = [scope.name for scope in dbutils.secrets.listScopes()]

In [0]:
w = WorkspaceClient()
enablement_pkg_scope = "enablement-package"

if enablement_pkg_scope in list_scope_names:
  print(f"{enablement_pkg_scope} scope already exists")
else:
    print(f"creating {enablement_pkg_scope} scope...")
    w.secrets.create_scope(scope=enablement_pkg_scope)

## Roboflow API key secret

In [0]:
roboflow_api_key_value = ""

if not w.secrets.get_secret(scope=enablement_pkg_scope, key="roboflow_api_key"):
  print("creating roboflow_api_key secret...")
  if roboflow_api_key_value:
    w.secrets.put_secret(scope=enablement_pkg_scope, key="roboflow_api_key", string_value=roboflow_api_key_value)
  else:
    print("The roboflow_api_key_value is empty.")
else:
  print("roboflow_api_key secret already exists")

## Grant reading access to a scope's secrets

In [0]:
user_credential = ""

In [0]:
if user_credential:
    print(f"Granting read access to {user_credential}...")
    w.secrets.put_acl(scope=enablement_pkg_scope, 
                      principal=user_credential, 
                      permission=AclPermission.READ)
else:
    print("No user credential provided. Skipping granting read access.")